# 16 · 군집 모델 구현과 해석 — 정답 없이 와인 묶기
교과서 **Ⅱ-04 k-평균(97쪽)** 을 코랩으로 직접 구현하고, Ⅱ-05 처럼 **결과를 평가 · 해석**합니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

**문제 정의** — 와인 178병의 화학 성분 13가지만 보고, **정답(품종)을 모르는 채로** 비슷한 와인끼리 묶는다.
정답을 쓰지 않음 → **비지도학습 · 군집화**. (정답은 마지막에 «채점용»으로만 꺼내 봅니다.)

| 단계 | 핵심 함수 |
|---|---|
| ① 불러오기 · 탐색 | `load_wine` `describe` |
| ② 전처리 — 단위 맞추기 | `StandardScaler().fit_transform` |
| ③ k 고르기 | `inertia_`(엘보) · `silhouette_score` |
| ④ 군집 학습 | `KMeans(n_clusters=k).fit_predict` |
| ⑤ 시각화 | `PCA(2)` · 애니메이션 |
| ⑥ 해석 | 군집별 평균 → 이름 붙이기 |
| ⑦ 평가 | 실제 품종과 비교 `crosstab` `adjusted_rand_score` |
| ⑧ 새 와인 | `predict` |

In [ ]:
# (그래프에 한글이 네모로 나오면 한 번만 실행)
!pip -q install koreanize-matplotlib
import koreanize_matplotlib

## ① 불러오기와 탐색

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
정답 = wine.target            # 0·1·2 품종 — 군집에는 쓰지 않고 마지막 채점에만!
print(X.shape)
X.describe().T[['mean', 'min', 'max']].round(2)

**읽기** — `proline` 은 수백~1600, `hue` 는 0.5~1.7. 열마다 **단위와 크기**가 크게 다릅니다.
k-평균은 **거리**로 묶으니, 그대로 두면 숫자가 큰 proline 하나가 묶음을 혼자 정해 버립니다.

---
## ② 전처리 — 표준화(단위 맞추기)
`StandardScaler` — 열마다 «평균 0, 표준편차 1»로 바꿉니다. (Ⅱ-02 의 «변환»)
💡 학습과 변환을 한 번에 → `fit_transform`

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
Z = scaler.____(X)
print(Z.mean(axis=0).round(2)[:5], Z.std(axis=0).round(2)[:5])

---
## ③ k 는 몇 개로? — 엘보 방법과 실루엣 점수
k 는 **사람이 정하는 하이퍼파라미터**(교과서 97쪽). 두 가지 단서로 고릅니다.
- **엘보(팔꿈치)** — k 를 늘릴수록 «점과 중심 사이 거리 제곱의 합»(`inertia_`)이 줄어드는데, **확 꺾이는 지점**이 알맞은 k
- **실루엣 점수** — «내 무리와는 가깝고, 옆 무리와는 먼 정도» (−1 ~ 1, **클수록 잘 묶임**)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
ks = range(2, 9)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Z, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ks, inertia, 'o-'); ax[0].set_xlabel('k'); ax[0].set_title('elbow (inertia)')
ax[1].plot(ks, sil, 'o-'); ax[1].set_xlabel('k'); ax[1].set_title('silhouette')
plt.show()
print('실루엣이 가장 큰 k :', list(ks)[int(np.argmax(sil))])

---
## ④ 군집 학습 — `fit_predict`
💡 무리 수 k → `n_clusters`

In [ ]:
km = KMeans(____=3, n_init=10, random_state=0)
무리 = km.fit_predict(Z)
print(pd.Series(무리).value_counts().sort_index())

---
## ⑤ 시각화 — 13차원을 2차원으로(PCA) · 애니메이션
열이 13개라 그대로는 못 그립니다. **PCA** 로 정보를 최대한 살린 **두 축**으로 줄여 그립니다(모델 도감 «주성분 분석»).

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(2)
P = pca.fit_transform(Z)
C = pca.transform(km.cluster_centers_)
plt.scatter(P[:, 0], P[:, 1], c=무리, cmap='viridis', s=18)
plt.scatter(C[:, 0], C[:, 1], c='red', marker='X', s=200, label='centers')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.title('k-means clusters')
plt.show()
print('두 축이 담은 정보 :', (pca.explained_variance_ratio_.sum() * 100).round(1), '%')

### (애니메이션) 교과서 97쪽 네 단계 — 초기 중심 → 할당 → 갱신 → 반복
2차원으로 줄인 점들 위에서 k-평균을 **한 걸음씩** 돌려 봅니다. ▶ 를 누르세요.

In [ ]:
from matplotlib import animation
from IPython.display import HTML
rng = np.random.default_rng(3)
cent = P[rng.choice(len(P), 3, replace=False)]      # ① 초기 중심(무작위)
frames = []
for it in range(8):
    d = ((P[:, None, :] - cent[None]) ** 2).sum(-1)
    lab = d.argmin(1)                                 # ② 할당: 가장 가까운 중심
    frames.append((cent.copy(), lab.copy()))
    new = np.array([P[lab == j].mean(0) for j in range(3)])   # ③ 갱신: 무리의 평균으로
    if np.allclose(new, cent): break                  # ④ 바뀌지 않으면 멈춤
    cent = new
fig, ax = plt.subplots(figsize=(6, 5))
def draw(i):
    ax.clear(); c, l = frames[i]
    ax.scatter(P[:, 0], P[:, 1], c=l, cmap='viridis', s=15)
    ax.scatter(c[:, 0], c[:, 1], c='red', marker='X', s=250)
    ax.set_title(f'iteration {i + 1}')
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=900)
plt.close()
HTML(anim.to_jshtml())

---
## ⑥ 해석 — 무리마다 평균을 보고 «이름» 붙이기
군집은 번호(0 · 1 · 2)만 줄 뿐, **뜻은 사람이 붙입니다.** 원래 단위의 평균을 비교해 보세요.

In [ ]:
X['무리'] = 무리
요약 = X.groupby('무리')[['alcohol', 'malic_acid', 'color_intensity', 'hue', 'flavanoids', 'proline']].mean().round(2)
요약

**이름 붙여 보기** — 예: «알코올·프롤린 높은 무리», «색이 옅고 알코올 낮은 무리», «색이 진하고 플라보노이드 낮은 무리» …

> 무리 0: ______ · 무리 1: ______ · 무리 2: ______

---
## ⑦ 평가 — 숨겨 둔 정답과 비교하기
비지도학습에는 원래 «정답률»이 없습니다. 이번엔 마침 정답(품종)이 있어 **얼마나 비슷하게 묶었는지** 확인만 해 봅니다.
- `pd.crosstab(무리, 정답)` — 행: 우리가 만든 무리 / 열: 실제 품종
- `adjusted_rand_score` — 두 묶음이 얼마나 일치하나 (1 완전 일치 · 0 우연 수준). **번호가 달라도 묶음이 같으면 1**

In [ ]:
from sklearn.metrics import adjusted_rand_score
print(pd.crosstab(무리, 정답, rownames=['무리'], colnames=['실제 품종']))
print('일치도(ARI) :', round(adjusted_rand_score(정답, 무리), 3))

In [ ]:
# 비교 실험 — 표준화를 안 하면?
무리2 = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X.drop(columns='무리'))
print('표준화 없이 ARI :', round(adjusted_rand_score(정답, 무리2), 3))

**생각해 보기** — 표준화를 했을 때와 안 했을 때 일치도가 얼마나 다른가요? 왜 그럴까요? (힌트: proline 의 숫자 크기)

---
## ⑧ 새 와인은 어느 무리?
새 데이터도 **같은 scaler 로 변환한 뒤** `predict` 합니다.

In [ ]:
새와인 = X.drop(columns='무리').iloc[[0, 70, 150]]      # 예시로 세 병을 다시 넣어 봄
print(km.predict(scaler.transform(새와인)))

---
## 결과 해석 — 문장으로
> 와인 178병을 화학 성분 13가지로 표준화한 뒤 k-평균(k=___)으로 묶었다. 실루엣 점수는 ___ 로 k=___ 일 때 가장 높았다.
> 무리 ___ 는 알코올과 프롤린이 높은 와인, 무리 ___ 는 … 이었다. 숨겨 둔 품종과 비교하니 일치도(ARI)가 ___ 로 **정답 없이도 품종을 거의 되찾았다.**
> 표준화를 하지 않으면 일치도가 ___ 로 떨어졌다 — 거리 기반 모델은 **단위 맞추기**가 필수다.

---
### 정답 예시 (막혔을 때만)
- ② `fit_transform` · ④ `n_clusters`
- k=3 에서 실루엣이 가장 크고, ARI 는 약 0.9 · 표준화 없이 약 0.37